# Juntar arquivos de raspagem → `base_armadilhas_concatenada`

Os 122 arquivos `.xlsx` em `Raspagem/Arquivos/` trazem **dados duplicados**: a mesma semana (`week_id`) aparece em vários arquivos (coletas em dias diferentes da semana). Para cada semana só interessa **o arquivo mais atualizado**, definido aqui como o que tem **mais mosquitos capturados** (desempate: data mais recente).

**Objetivo:** montar um único histórico de capturas ao longo do tempo, ficando apenas com o arquivo escolhido de cada semana, e salvar em `output/base_armadilhas_concatenada.csv` — que o `modelo1.ipynb` vai consumir no lugar dos 122 arquivos.

Construção **passo a passo**, um bloco por vez.

## Bloco 1 — indexar os arquivos e localizar as duplicatas

Lê apenas o **nome** de cada arquivo (não abre os Excel ainda) para extrair `data_coleta`, `week_id` e o nº de mosquitos embutido no nome. Em seguida mostra quantas semanas têm mais de um arquivo — são essas que serão deduplicadas.

Padrão do nome: `dados_aedes_<AAAAMMDD>_weekid<NN>_<N>mosquitos[_<k>].xlsx`

In [ ]:
import glob
import os
import re
from pathlib import Path

import pandas as pd


# Acha a raiz do projeto (Meu_Projeto/) subindo a partir do diretório atual
# até encontrar a pasta 'Raspagem' (evita caminho absoluto hard-coded).
def achar_raiz(marcador="Raspagem"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marcador).is_dir():
            return p
    raise FileNotFoundError(f"Pasta-raiz contendo '{marcador}/' nao encontrada a partir de {Path.cwd()}")


RAIZ = achar_raiz()
RASPAGEM_DIR = RAIZ / "Raspagem" / "Arquivos"

arquivos = sorted(glob.glob(str(RASPAGEM_DIR / "dados_aedes_*.xlsx")))

# Extrai do NOME: data da coleta (AAAAMMDD), week_id e total de mosquitos.
PADRAO = re.compile(r"dados_aedes_(\d{8})_weekid(\d+)_(\d+)mosquitos")

linhas = []
for caminho in arquivos:
    nome = os.path.basename(caminho)
    m = PADRAO.search(nome)
    if not m:
        print("NAO casou com o padrao:", nome)
        continue
    data_str, week_id, mosquitos = m.groups()
    linhas.append({
        "arquivo": nome,
        "data_coleta": pd.to_datetime(data_str, format="%Y%m%d"),
        "week_id": int(week_id),
        "mosquitos_nome": int(mosquitos),
    })

indice = pd.DataFrame(linhas).sort_values(["week_id", "data_coleta"]).reset_index(drop=True)

print(f"{len(indice)} arquivos indexados")
print(f"{indice['week_id'].nunique()} semanas (week_id) distintas")

dups = indice.groupby("week_id").size()
print(f"semanas com mais de 1 arquivo: {(dups > 1).sum()}  (serao deduplicadas)")
print(f"semanas com apenas 1 arquivo:  {(dups == 1).sum()}")
indice

## Bloco 2 — abrir cada arquivo e RECONTAR os mosquitos (verificação)

Abre os 122 Excel, soma a coluna `total_mosquitos` de cada um (contagem real, sem depender do nome) e guarda os DataFrames em memória para reusar na concatenação. A coluna `confere_nome` confirma que o total real bate com o número do nome do arquivo.

In [ ]:
dfs_por_arquivo = {}
contagens = []
for _, lin in indice.iterrows():
    df = pd.read_excel(RASPAGEM_DIR / lin["arquivo"])
    dfs_por_arquivo[lin["arquivo"]] = df
    contagens.append(int(df["total_mosquitos"].sum()))

indice["mosquitos_real"] = contagens
indice["confere_nome"] = indice["mosquitos_real"] == indice["mosquitos_nome"]

print(f"{len(indice)} arquivos abertos e recontados")
print(f"total real == total do nome em {int(indice['confere_nome'].sum())}/{len(indice)} arquivos")
indice[["arquivo", "week_id", "data_coleta", "mosquitos_nome", "mosquitos_real", "confere_nome"]]

## Bloco 3 — selecionar 1 arquivo por semana (o de mais mosquitos)

Para cada `week_id`, fica com o arquivo de **maior `mosquitos_real`**; em caso de empate, o de **data mais recente**. Resultado: uma linha por semana.

In [ ]:
selecionados = (
    indice
    .sort_values(["week_id", "mosquitos_real", "data_coleta"], ascending=[True, False, False])
    .drop_duplicates(subset="week_id", keep="first")
    .sort_values("data_coleta")
    .reset_index(drop=True)
)

print(f"{len(selecionados)} semanas selecionadas (1 arquivo por semana)")
print(f"{len(indice) - len(selecionados)} arquivos descartados (duplicatas de semana)")
selecionados[["week_id", "data_coleta", "arquivo", "mosquitos_real"]]

## Bloco 4 — concatenar as linhas dos arquivos selecionados

Empilha as armadilhas (1 linha = 1 inspeção) só dos arquivos escolhidos, adicionando `arquivo_origem` e `data_coleta` para rastreabilidade. Esta é a `base_armadilhas` (nível armadilha, já deduplicada por semana).

In [ ]:
partes = []
for _, lin in selecionados.iterrows():
    df = dfs_por_arquivo[lin["arquivo"]].copy()
    df["arquivo_origem"] = lin["arquivo"]
    df["data_coleta"] = lin["data_coleta"]
    partes.append(df)

# checagem: todos os arquivos têm as mesmas colunas?
colunas_distintas = {tuple(p.columns) for p in partes}
print("schemas distintos entre os arquivos:", len(colunas_distintas), "(esperado: 1)")

base = pd.concat(partes, ignore_index=True)
print("linhas x colunas:", base.shape)
print("semanas (week_id):", base["week_id"].nunique())
print("período das coletas:", base["data_coleta"].min().date(), "->", base["data_coleta"].max().date())
base.head()

## Bloco 5 — salvar `base_armadilhas_concatenada.csv`

Grava o resultado em `output/`. É este arquivo que o `modelo1.ipynb` vai ler no lugar dos 122 `.xlsx`.

In [ ]:
OUTPUT_DIR = RAIZ / "Bases de dados" / "juntar_arquivos_raspagem" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destino = OUTPUT_DIR / "base_armadilhas_concatenada.csv"

base.to_csv(destino, index=False)
print("salvo em:", destino)
print("linhas x colunas:", base.shape)
print("tamanho:", round(destino.stat().st_size / 1_000_000, 2), "MB")